# Store Impression Correlation and Silhouette Analysis

This notebook analyzes store impression data to:
- Calculate correlations between stores based on daily impressions
- Perform clustering using K-Means
- Calculate silhouette scores to evaluate clustering quality
- Visualize store groupings and patterns

## Data Dictionary
- **f0_** (date): Date of the impression data
- **zo_name**: Zone name (京浜)
- **do_cd**: District code (store_cd at district level)
- **do_name**: District name
- **store_cd**: Unique store code
- **store_name_kanji**: Store name in Japanese
- **f1_** (daily_impression): Daily impression count
- **FileName**: Source file name

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

# Add project root to path to ensure imports work correctly
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"Project root: {project_root}")
print(f"Current working directory: {os.getcwd()}")
print(f"Python version: {sys.version}")

## 1. Load and Explore Data

In [ ]:
# Load the store impression data
data_path = os.path.join(project_root, 'data', 'store_impression_data.csv')
df = pd.read_csv(data_path)

# Rename columns for easier reference
df.rename(columns={
    'f0_': 'date',
    'f1_': 'daily_impression'
}, inplace=True)

print(f"Loaded {len(df)} records")
print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"Number of unique stores: {df['store_cd'].nunique()}")
print(f"Number of unique districts: {df['do_cd'].nunique()}")
df.head(10)

In [ ]:
# Convert date to datetime format
df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y')

# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Basic statistics
print("\nDaily Impression Statistics:")
print(df['daily_impression'].describe())

## 2. Data Preprocessing and Feature Engineering

In [ ]:
# Create a pivot table: stores as rows, dates as columns, impressions as values
# This allows us to analyze patterns across stores
store_impression_pivot = df.pivot_table(
    index='store_cd',
    columns='date',
    values='daily_impression',
    aggfunc='sum'
).fillna(0)

print(f"Pivot table shape: {store_impression_pivot.shape}")
print(f"Stores (rows): {store_impression_pivot.shape[0]}")
print(f"Dates (columns): {store_impression_pivot.shape[1]}")
print("\nPivot table preview:")
store_impression_pivot.head()

In [ ]:
# Create store metadata for later use
store_metadata = df[['store_cd', 'store_name_kanji', 'do_name', 'do_cd', 'zo_name']].drop_duplicates()
store_metadata.set_index('store_cd', inplace=True)

print("Store metadata:")
store_metadata.head(10)

In [ ]:
# Calculate aggregate features per store
store_features = df.groupby('store_cd').agg({
    'daily_impression': ['mean', 'std', 'min', 'max', 'sum'],
    'date': 'count'
}).reset_index()

store_features.columns = ['store_cd', 'avg_impression', 'std_impression', 
                           'min_impression', 'max_impression', 'total_impression', 'num_records']

# Calculate coefficient of variation (CV) - measure of relative variability
store_features['cv_impression'] = store_features['std_impression'] / store_features['avg_impression']
store_features['cv_impression'] = store_features['cv_impression'].replace([np.inf, -np.inf], 0).fillna(0)

# Merge with metadata
store_features = store_features.merge(store_metadata, on='store_cd', how='left')

print("Store features:")
store_features.head(10)

## 3. Correlation Analysis Between Stores

In [ ]:
# Calculate correlation matrix between stores based on their impression patterns
store_correlation = store_impression_pivot.T.corr()

print(f"Correlation matrix shape: {store_correlation.shape}")
print("\nCorrelation matrix preview:")
print(store_correlation.head())

# Summary statistics of correlations
# Extract upper triangle (excluding diagonal) to avoid duplicates
corr_values = store_correlation.values[np.triu_indices_from(store_correlation.values, k=1)]
print(f"\nCorrelation statistics (between different stores):")
print(f"Mean correlation: {corr_values.mean():.4f}")
print(f"Median correlation: {np.median(corr_values):.4f}")
print(f"Std correlation: {corr_values.std():.4f}")
print(f"Min correlation: {corr_values.min():.4f}")
print(f"Max correlation: {corr_values.max():.4f}")

In [ ]:
# Visualize correlation matrix
plt.figure(figsize=(14, 12))
sns.heatmap(store_correlation, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1)
plt.title('Store Correlation Matrix Based on Daily Impressions', fontsize=16, pad=20)
plt.xlabel('Store Code', fontsize=12)
plt.ylabel('Store Code', fontsize=12)
plt.tight_layout()
plt.show()

print("Heatmap shows correlation between stores:")
print("- Red indicates positive correlation (similar patterns)")
print("- Blue indicates negative correlation (opposite patterns)")
print("- White indicates no correlation")

In [ ]:
# Find most correlated store pairs
# Create a dataframe of store pairs and their correlations
store_pairs = []
stores = store_correlation.index.tolist()

for i in range(len(stores)):
    for j in range(i+1, len(stores)):
        store_pairs.append({
            'store_1': stores[i],
            'store_2': stores[j],
            'correlation': store_correlation.iloc[i, j]
        })

store_pairs_df = pd.DataFrame(store_pairs)

# Add store names
store_pairs_df = store_pairs_df.merge(
    store_metadata[['store_name_kanji']], 
    left_on='store_1', 
    right_index=True
).rename(columns={'store_name_kanji': 'store_1_name'})

store_pairs_df = store_pairs_df.merge(
    store_metadata[['store_name_kanji']], 
    left_on='store_2', 
    right_index=True
).rename(columns={'store_name_kanji': 'store_2_name'})

# Top 10 most positively correlated stores
print("Top 10 Most Positively Correlated Store Pairs:")
print("=" * 100)
print(store_pairs_df.nlargest(10, 'correlation')[['store_1', 'store_1_name', 'store_2', 'store_2_name', 'correlation']])

# Top 10 most negatively correlated stores
print("\nTop 10 Most Negatively Correlated Store Pairs:")
print("=" * 100)
print(store_pairs_df.nsmallest(10, 'correlation')[['store_1', 'store_1_name', 'store_2', 'store_2_name', 'correlation']])

## 4. Clustering Analysis with K-Means

In [ ]:
# Prepare features for clustering
# We'll use the impression patterns (pivot table) as features
X = store_impression_pivot.values

# Standardize features (important for K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Features are standardized (mean≈0, std≈1)")
print(f"Mean: {X_scaled.mean():.4f}")
print(f"Std: {X_scaled.std():.4f}")

In [ ]:
# Determine optimal number of clusters using elbow method and silhouette score
K_range = range(2, min(11, len(X_scaled)))  # Test 2 to 10 clusters (or less if we have fewer stores)
inertias = []
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Elbow curve
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia (Within-cluster Sum of Squares)', fontsize=12)
axes[0].set_title('Elbow Method For Optimal K', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Silhouette scores
axes[1].plot(K_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score For Different K', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

# Find optimal K
optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"\nOptimal number of clusters based on silhouette score: {optimal_k}")
print(f"Maximum silhouette score: {max(silhouette_scores):.4f}")
print("\nSilhouette scores for each K:")
for k, score in zip(K_range, silhouette_scores):
    print(f"K={k}: {score:.4f}")

## 5. Silhouette Score Calculation Per Store

In [ ]:
# Perform clustering with optimal K
kmeans_optimal = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans_optimal.fit_predict(X_scaled)

# Calculate silhouette scores for each store
silhouette_vals = silhouette_samples(X_scaled, cluster_labels)

# Create results dataframe
clustering_results = pd.DataFrame({
    'store_cd': store_impression_pivot.index,
    'cluster': cluster_labels,
    'silhouette_score': silhouette_vals
})

# Merge with store features and metadata
clustering_results = clustering_results.merge(store_features, on='store_cd', how='left')

# Sort by cluster and silhouette score
clustering_results = clustering_results.sort_values(['cluster', 'silhouette_score'], ascending=[True, False])

print(f"Overall Silhouette Score: {silhouette_score(X_scaled, cluster_labels):.4f}")
print(f"\nNumber of stores per cluster:")
print(clustering_results['cluster'].value_counts().sort_index())
print("\nClustering results with silhouette scores:")
print(clustering_results[['store_cd', 'store_name_kanji', 'cluster', 'silhouette_score', 
                           'avg_impression', 'total_impression']].head(20))

In [ ]:
# Analyze silhouette scores by cluster
cluster_silhouette_stats = clustering_results.groupby('cluster')['silhouette_score'].agg([
    'count', 'mean', 'std', 'min', 'max'
]).round(4)

print("Silhouette Score Statistics by Cluster:")
print("=" * 80)
print(cluster_silhouette_stats)

# Identify stores with negative silhouette scores (poorly clustered)
poorly_clustered = clustering_results[clustering_results['silhouette_score'] < 0]
print(f"\nStores with negative silhouette scores (poorly clustered): {len(poorly_clustered)}")
if len(poorly_clustered) > 0:
    print(poorly_clustered[['store_cd', 'store_name_kanji', 'cluster', 'silhouette_score', 'avg_impression']])

## 6. Visualization of Clustering Results

In [ ]:
# Silhouette plot
fig, ax = plt.subplots(figsize=(12, 8))

y_lower = 10
for i in range(optimal_k):
    # Get silhouette scores for cluster i
    cluster_silhouette_vals = silhouette_vals[cluster_labels == i]
    cluster_silhouette_vals.sort()
    
    size_cluster_i = cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    color = plt.cm.nipy_spectral(float(i) / optimal_k)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_vals,
                     facecolor=color, edgecolor=color, alpha=0.7)
    
    # Label the silhouette plots with their cluster numbers at the middle
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i), fontsize=14, fontweight='bold')
    
    y_lower = y_upper + 10

ax.set_title('Silhouette Plot for Store Clusters', fontsize=16, pad=20)
ax.set_xlabel('Silhouette Coefficient', fontsize=13)
ax.set_ylabel('Cluster Label', fontsize=13)

# Vertical line for average silhouette score
ax.axvline(x=silhouette_score(X_scaled, cluster_labels), color="red", linestyle="--", 
           linewidth=2, label=f'Average Score: {silhouette_score(X_scaled, cluster_labels):.3f}')
ax.axvline(x=0, color="black", linestyle="-", linewidth=1, alpha=0.3)

ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Wider silhouette plots indicate larger clusters")
print("- Scores close to 1 indicate well-separated clusters")
print("- Scores close to 0 indicate overlapping clusters")
print("- Negative scores indicate potentially misclassified stores")

In [ ]:
# Box plot of silhouette scores by cluster
plt.figure(figsize=(12, 6))
clustering_results.boxplot(column='silhouette_score', by='cluster', figsize=(12, 6))
plt.suptitle('')  # Remove default title
plt.title('Distribution of Silhouette Scores by Cluster', fontsize=14, pad=20)
plt.xlabel('Cluster', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Zero line')
plt.axhline(y=silhouette_score(X_scaled, cluster_labels), color='green', 
            linestyle='--', alpha=0.5, label='Average score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Average Impression vs Silhouette Score
plt.figure(figsize=(14, 8))
for cluster in range(optimal_k):
    cluster_data = clustering_results[clustering_results['cluster'] == cluster]
    plt.scatter(cluster_data['avg_impression'], 
                cluster_data['silhouette_score'],
                label=f'Cluster {cluster}',
                s=100, alpha=0.6, edgecolors='black', linewidth=0.5)

plt.xlabel('Average Daily Impression', fontsize=13)
plt.ylabel('Silhouette Score', fontsize=13)
plt.title('Store Clustering: Average Impression vs Silhouette Score', fontsize=16, pad=20)
plt.axhline(y=0, color='red', linestyle='--', alpha=0.3)
plt.axhline(y=silhouette_score(X_scaled, cluster_labels), color='green', 
            linestyle='--', alpha=0.3, label='Avg Silhouette')
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Cluster Profiling and Interpretation

In [ ]:
# Profile each cluster
cluster_profiles = clustering_results.groupby('cluster').agg({
    'store_cd': 'count',
    'avg_impression': ['mean', 'std', 'min', 'max'],
    'total_impression': ['mean', 'sum'],
    'cv_impression': 'mean',
    'silhouette_score': ['mean', 'std', 'min', 'max']
}).round(2)

cluster_profiles.columns = ['_'.join(col).strip('_') for col in cluster_profiles.columns.values]
cluster_profiles.rename(columns={'store_cd_count': 'num_stores'}, inplace=True)

print("Cluster Profiles:")
print("=" * 120)
print(cluster_profiles)

# Save cluster profiles
output_path = os.path.join(project_root, 'data', 'cluster_profiles.csv')
cluster_profiles.to_csv(output_path)
print(f"\nCluster profiles saved to: {output_path}")

In [ ]:
# Show representative stores from each cluster (highest silhouette scores)
print("Representative Stores from Each Cluster (Top 3 by Silhouette Score):")
print("=" * 120)
for cluster in range(optimal_k):
    print(f"\nCluster {cluster}:")
    print("-" * 120)
    cluster_data = clustering_results[clustering_results['cluster'] == cluster]
    top_stores = cluster_data.nlargest(3, 'silhouette_score')
    print(top_stores[['store_cd', 'store_name_kanji', 'do_name', 'silhouette_score', 
                       'avg_impression', 'total_impression']].to_string(index=False))

In [ ]:
# Visualize cluster characteristics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Number of stores per cluster
cluster_counts = clustering_results['cluster'].value_counts().sort_index()
axes[0, 0].bar(cluster_counts.index, cluster_counts.values, color='skyblue', edgecolor='black')
axes[0, 0].set_xlabel('Cluster', fontsize=12)
axes[0, 0].set_ylabel('Number of Stores', fontsize=12)
axes[0, 0].set_title('Store Distribution Across Clusters', fontsize=13)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 2. Average impression by cluster
cluster_avg_impressions = clustering_results.groupby('cluster')['avg_impression'].mean()
axes[0, 1].bar(cluster_avg_impressions.index, cluster_avg_impressions.values, 
               color='lightcoral', edgecolor='black')
axes[0, 1].set_xlabel('Cluster', fontsize=12)
axes[0, 1].set_ylabel('Average Daily Impression', fontsize=12)
axes[0, 1].set_title('Average Impression by Cluster', fontsize=13)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Average silhouette score by cluster
cluster_avg_silhouette = clustering_results.groupby('cluster')['silhouette_score'].mean()
colors = ['green' if x > 0 else 'red' for x in cluster_avg_silhouette.values]
axes[1, 0].bar(cluster_avg_silhouette.index, cluster_avg_silhouette.values, 
               color=colors, edgecolor='black', alpha=0.7)
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[1, 0].axhline(y=silhouette_score(X_scaled, cluster_labels), color='blue', 
                   linestyle='--', linewidth=2, label='Overall Average')
axes[1, 0].set_xlabel('Cluster', fontsize=12)
axes[1, 0].set_ylabel('Average Silhouette Score', fontsize=12)
axes[1, 0].set_title('Cluster Quality (Silhouette Score)', fontsize=13)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Coefficient of variation by cluster
cluster_cv = clustering_results.groupby('cluster')['cv_impression'].mean()
axes[1, 1].bar(cluster_cv.index, cluster_cv.values, color='lightgreen', edgecolor='black')
axes[1, 1].set_xlabel('Cluster', fontsize=12)
axes[1, 1].set_ylabel('Average CV (Std/Mean)', fontsize=12)
axes[1, 1].set_title('Impression Variability by Cluster', fontsize=13)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Hierarchical Clustering (Dendrogram)

In [ ]:
# Perform hierarchical clustering
linkage_matrix = linkage(X_scaled, method='ward')

# Create dendrogram
plt.figure(figsize=(16, 8))
dendrogram(linkage_matrix, labels=store_impression_pivot.index.tolist(),
           leaf_rotation=90, leaf_font_size=8)
plt.title('Hierarchical Clustering Dendrogram of Stores', fontsize=16, pad=20)
plt.xlabel('Store Code', fontsize=13)
plt.ylabel('Distance (Ward)', fontsize=13)
plt.axhline(y=linkage_matrix[-optimal_k+1, 2], color='red', linestyle='--', 
            linewidth=2, label=f'Cut for {optimal_k} clusters')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Dendrogram shows hierarchical relationships between stores")
print(f"Red line indicates the cut height for {optimal_k} clusters")

## 9. Save Results

In [ ]:
# Save clustering results with silhouette scores
output_path = os.path.join(project_root, 'data', 'store_clustering_results.csv')
clustering_results.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Clustering results saved to: {output_path}")

# Save correlation matrix
corr_output_path = os.path.join(project_root, 'data', 'store_correlation_matrix.csv')
store_correlation.to_csv(corr_output_path, encoding='utf-8-sig')
print(f"Correlation matrix saved to: {corr_output_path}")

# Save store pairs with correlations
pairs_output_path = os.path.join(project_root, 'data', 'store_pairs_correlation.csv')
store_pairs_df.to_csv(pairs_output_path, index=False, encoding='utf-8-sig')
print(f"Store pairs correlation saved to: {pairs_output_path}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)
print(f"Total stores analyzed: {len(clustering_results)}")
print(f"Optimal number of clusters: {optimal_k}")
print(f"Overall silhouette score: {silhouette_score(X_scaled, cluster_labels):.4f}")
print(f"Average correlation between stores: {corr_values.mean():.4f}")

## 10. Summary and Insights

In [ ]:
print("="*100)
print("STORE IMPRESSION ANALYSIS SUMMARY")
print("="*100)

print("\n1. DATA OVERVIEW:")
print(f"   - Total stores: {len(clustering_results)}")
print(f"   - Date range: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
print(f"   - Average daily impression per store: {clustering_results['avg_impression'].mean():.0f}")
print(f"   - Total impressions across all stores: {clustering_results['total_impression'].sum():.0f}")

print("\n2. CORRELATION ANALYSIS:")
print(f"   - Average correlation between stores: {corr_values.mean():.4f}")
print(f"   - Highest correlation: {corr_values.max():.4f}")
print(f"   - Lowest correlation: {corr_values.min():.4f}")

print("\n3. CLUSTERING RESULTS:")
print(f"   - Optimal number of clusters: {optimal_k}")
print(f"   - Overall silhouette score: {silhouette_score(X_scaled, cluster_labels):.4f}")
print(f"   - Stores with negative silhouette scores: {len(poorly_clustered)}")

print("\n4. CLUSTER BREAKDOWN:")
for cluster in range(optimal_k):
    cluster_data = clustering_results[clustering_results['cluster'] == cluster]
    print(f"   Cluster {cluster}:")
    print(f"      - Number of stores: {len(cluster_data)}")
    print(f"      - Avg impression: {cluster_data['avg_impression'].mean():.0f}")
    print(f"      - Avg silhouette score: {cluster_data['silhouette_score'].mean():.4f}")
    print(f"      - Top store: {cluster_data.nlargest(1, 'silhouette_score')['store_name_kanji'].values[0]}")

print("\n5. KEY INSIGHTS:")
if silhouette_score(X_scaled, cluster_labels) > 0.5:
    print("   ✓ Strong clustering structure - stores have distinct impression patterns")
elif silhouette_score(X_scaled, cluster_labels) > 0.25:
    print("   ✓ Moderate clustering structure - stores show some grouping patterns")
else:
    print("   ⚠ Weak clustering structure - stores have overlapping patterns")

high_corr_count = len(store_pairs_df[store_pairs_df['correlation'] > 0.7])
if high_corr_count > 0:
    print(f"   ✓ {high_corr_count} store pairs have high correlation (>0.7) - similar performance patterns")

print("\n" + "="*100)